### Paragraph (Long Text) Vector Search

### What This Notebook Does

This notebook builds a **vector-based similarity search engine** for long text documents. It:

1. **Splits** long documents into smaller chunks (paragraph-sized pieces).
2. **Converts** each chunk into a numerical vector using **TF-IDF** (Term Frequency-Inverse Document Frequency).
3. **Stores** all chunk vectors in an in-memory vector database.
4. **Accepts a query** (e.g., "What is a Vector Database?") and converts it to a TF-IDF vector.
5. **Computes cosine similarity** between the query vector and all chunk vectors.
6. **Returns** the top-k most similar chunks.

### Why This Is Useful

This notebook (TF-IDF vector search) is the **building block** for these real-world use cases:

1. **RAG (Retrieval-Augmented Generation)** — The most common use. Given a user question, find relevant chunks from a knowledge base → pass them to an LLM as context → the LLM answers based on those chunks. The vector search is the "Retrieval" in RAG.

2. **Document Search / Q&A over company docs** — Upload PDFs/notes, chunk them, and let users ask questions (e.g. "What's our return policy?"). The vector search finds the right paragraphs.

3. **Plagiarism / similarity check** — Check if a new piece of text is similar to existing content by comparing their vectors.

4. **Recommendation systems** — Find similar items based on their text descriptions (e.g. "show me product descriptions similar to this one").

5. **Code search** — Search codebases by converting code comments/docs to vectors and matching against natural language queries.

**Note:** For semantic understanding (e.g. "car" ≈ "vehicle"), you'd replace TF-IDF with **dense embeddings** (like Sentence-BERT or OpenAI embeddings). But the chunking + vector search pattern remains the same — that's why this notebook teaches the core architecture.

Would you like me to add this explanation into `1_para_search2.ipynb` as a markdown cell?

In [1]:
# Import required libraries:
#   numpy: for numerical operations
#   TfidfVectorizer: converts text into TF-IDF vectors
#   cosine_similarity: measures similarity between vectors
#   vstack: stacks sparse matrices vertically
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import vstack

## LongTextVectorSearch Class

This class encapsulates the full vector search workflow:
- **`__init__`**: Sets up the vectorizer, vector database (list), chunk size, and chunk text tracker.
- **`split_text`**: Breaks a long document into chunks of `chunk_size` words each.
- **`create_vector_db`**: Takes a list of texts, splits them, fits the TF-IDF vectorizer, and converts all chunks to vectors.
- **`search_similar_texts`**: Converts a query to a vector, computes cosine similarity against all stored chunk vectors, and returns the top-k matches.

In [2]:
class LongTextVectorSearch:
    """
    Performs vector-based similarity search on long text documents.

    Workflow:
    1. Split long documents into smaller chunks.
    2. Convert each chunk into a TF-IDF vector.
    3. Store all vectors in a vector database.
    4. Convert the query into a vector.
    5. Compute cosine similarity between the query and all chunks.
    6. Return the most similar chunks.
    """

    def __init__(self, chunk_size=100):
        """
        Initialize the vector search system.
        
        Args:
            chunk_size (int): Number of words per chunk when splitting long text.
                              Smaller chunks = more granular but more vectors.
        """

        # Create the TF-IDF vectorizer
        # TF-IDF = Term Frequency - Inverse Document Frequency
        # It converts text into a numerical vector where:
        # - Frequent words in a document get higher weight
        # - Words that appear across many documents get lower weight (less discriminating)
        self.vectorizer = TfidfVectorizer()

        # Stores the vectors of all text chunks as a sparse matrix
        # Each row = one chunk's TF-IDF vector
        # Each column = one vocabulary word's TF-IDF score
        self.vector_db = []

        # Number of words in each chunk
        # Controls granularity: smaller chunks capture more specific context
        self.chunk_size = chunk_size

        # Stores the chunk identifiers (e.g., "text_0_chunk_3")
        # Keeps track of which chunk came from which original document
        self.chunk_texts = []

    def split_text(self, text):
        """
        Split a long document into smaller chunks.
        
        This is essential because:
        - TF-IDF works best on paragraph-sized text, not entire books
        - RAG systems retrieve specific chunks, not entire documents
        - Smaller chunks = more precise search results
        
        Args:
            text (str): The full document text to split.
            
        Returns:
            list[str]: List of text chunks, each with 'chunk_size' words.
        """

        # Split the document into individual words (by whitespace)
        words = text.split()

        # Create chunks of 'chunk_size' words using a sliding window approach
        # range(0, len(words), self.chunk_size) steps through the word list
        # in increments of chunk_size, creating non-overlapping chunks
        return [
            ' '.join(words[i:i + self.chunk_size])  # Join chunk_size words back into text
            for i in range(0, len(words), self.chunk_size)  # Step through word indices
        ]

    def create_vector_db(self, texts):
        """
        Create a vector database from multiple text documents.
        
        This method:
        1. Splits each document into chunks
        2. Fits the TF-IDF vectorizer on all chunks (learns vocabulary)
        3. Transforms all chunks into TF-IDF vectors
        4. Stores them in the in-memory vector database
        
        Args:
            texts (list[str]): List of long documents to index.
        """

        # Stores all text chunks from all documents
        all_chunks = []

        # Process each document one at a time
        for i, text in enumerate(texts):

            # Split the document into chunks of 'chunk_size' words
            chunks = self.split_text(text)

            # Add the chunks to the master list (flat list of all chunks)
            all_chunks.extend(chunks)

            # Create a unique ID for every chunk of the form "text_{doc_index}_chunk_{chunk_index}"
            # This allows us to trace back which original document a chunk came from
            self.chunk_texts.extend([
                f"text_{i}_chunk_{j}"
                for j in range(len(chunks))
            ])

            print(f"Added {len(chunks)} chunks for text_{i}")

        # Step 1: Fit the vectorizer on all chunks to learn the vocabulary
        # This builds the TF-IDF matrix where:
        # - Rows = chunks
        # - Columns = unique words (vocabulary)
        # - Values = TF-IDF scores
        self.vectorizer.fit(all_chunks)

        # Step 2: Transform every chunk into a TF-IDF vector
        # This creates a sparse matrix: each row is a chunk's vector representation
        # The vector captures which words are important in each chunk
        self.vector_db = self.vectorizer.transform(all_chunks)

        # Display some statistics about the vector database
        print(f"Total chunks: {len(self.chunk_texts)}")
        print(f"Vector shape: {self.vector_db.shape}")  # (chunks, vocabulary_size)
        print(f"Vocabulary size: {len(self.vectorizer.get_feature_names_out())}")

    def search_similar_texts(self, query_text, top_k=5):
        """
        Search for the most similar text chunks given a query.
        
        This is the retrieval step — the core of RAG.
        It finds chunks that are semantically similar to the query
        based on TF-IDF vector cosine similarity.
        
        Args:
            query_text (str): The search query.
            top_k (int): Number of top results to return.
            
        Returns:
            list[tuple[str, float]]: List of (chunk_id, similarity_score) tuples,
                                     sorted by similarity (highest first).
        """

        # Convert the query into a TF-IDF vector using the same vocabulary
        # This ensures the query vector is in the same "space" as our chunk vectors
        query_vector = self.vectorizer.transform([query_text])

        print(f"Generated query vector for: '{query_text[:50]}...'")

        # Calculate cosine similarity between the query
        # and every chunk in the vector database
        # cosine_similarity returns a 2D array; .flatten() makes it 1D
        # cos(A, B) = (A · B) / (||A|| * ||B||)
        #   = 1.0: identical vectors (perfect match)
        #   = 0.0: orthogonal (no similarity)
        #   = -1.0: opposite (rare in TF-IDF)
        similarities = cosine_similarity(
            query_vector,
            self.vector_db
        ).flatten()

        print(similarities[:5])  # Show first 5 similarity scores for debugging

        # Get the indices of the top-k most similar chunks
        # argsort() returns indices sorted by similarity (ascending)
        # [-top_k:] gets the last k indices (highest similarity)
        # [::-1] reverses to get descending order (highest first)
        top_indices = similarities.argsort()[-top_k:][::-1]

        # Store the search results as (chunk_id, similarity_score) tuples
        results = []

        # Retrieve the chunk IDs and similarity scores
        for index in top_indices:

            # Chunk identifier (e.g., "text_0_chunk_5")
            chunk_id = self.chunk_texts[index]

            # Similarity score between 0 and 1
            similarity = similarities[index]

            # Add to results list
            results.append((chunk_id, similarity))

            print(
                f"Similarity between query and "
                f"{chunk_id}: {similarity:.4f}"
            )

        # Return the top matching chunks (highest similarity first)
        return results

## Example Data

We'll use a long document about **Vector Databases** (~650 words) as our test data. This document covers:
- What a vector database is
- Comparison with traditional databases
- Applications across industries (e-commerce, healthcare, finance, etc.)

The document will be split into chunks of 50 words each (~13 chunks total).

In [3]:

# Example with longer texts
texts = [
"""What Is a Vector Database?

A vector database is a specialized data storage and retrieval system designed to handle high-dimensional vector data efficiently. In the context of machine learning and artificial intelligence, vectors are numerical representations of data objects, such as words, images, or user behaviors, encoded in a multi-dimensional space. These vectors, often called embeddings, capture the semantic or contextual meaning of the data.

Key Characteristics:

Storage of High-Dimensional Data: Capable of storing vectors with hundreds or thousands of dimensions.
Efficient Similarity Search: Optimized for operations like nearest neighbor search, which finds vectors similar to a given query vector.
Scalability: Designed to handle large volumes of data, potentially billions of vectors.
Integration with AI Models: Works seamlessly with machine learning models that generate embeddings.
Differences Between Vector Databases and Traditional Databases

Data Representation:

Traditional Databases: Store structured data (tables with rows and columns) or unstructured data (documents).
Vector Databases: Store data as high-dimensional vectors representing complex data relationships.
Query Mechanisms:

Traditional Databases: Use SQL queries or key-value lookups for exact matches or range queries.
Vector Databases: Perform similarity searches using distance metrics (e.g., cosine similarity, Euclidean distance).
Indexing Structures:

Traditional Databases: Use B-trees, hash indexes, or inverted indexes.
Vector Databases: Employ specialized indexes like k-d trees, VP-trees, or approximate nearest neighbor algorithms.
Performance Optimization:

Traditional Databases: Optimize for transaction throughput and ACID properties.
Vector Databases: Optimize for low-latency similarity searches over large datasets.
Importance and Applications of Vector Databases

Handling Complex Data: Enable the storage and retrieval of data types that are difficult to manage with traditional databases (e.g., images, audio, text semantics).
Real-Time Recommendations: Provide immediate suggestions based on user interactions by quickly finding similar items.
Enhanced Search Capabilities: Allow semantic search, where queries return results based on meaning rather than keyword matching.
Machine Learning Integration: Serve as a backend for AI applications that require fast access to vector representations.
Use Cases in Industries

E-commerce:

Product Recommendations: Suggest products similar to those a user has viewed or purchased.
Visual Search: Allow users to search for products using images.
Media and Entertainment:

Content Recommendations: Suggest movies, songs, or articles based on user preferences.
Image and Video Retrieval: Find visually similar media content.
Healthcare:

Medical Imaging: Compare patient scans to identify anomalies or similar cases.
Genomics: Analyze genetic data represented as vectors.
Finance:

Fraud Detection: Identify unusual transactions by comparing transaction vectors.
Risk Assessment: Evaluate investment similarities and portfolio diversification."""  # ~650 words
]

## Build the Vector Database

Here we:
1. Create an instance of `LongTextVectorSearch` with `chunk_size=50` (50 words per chunk)
2. Call `create_vector_db()` which:
   - Splits the long text into ~13 chunks
   - Fits the TF-IDF vectorizer on all chunks
   - Transforms each chunk into a vector
   - Reports statistics (total chunks, vector shape, vocabulary size)

In [4]:
# Create the vector search engine with 50-word chunks
# Smaller chunk size = more precise retrieval but more vectors to search
search_engine = LongTextVectorSearch(chunk_size=50)  # 50 words per chunk

print("Creating vector database...")
search_engine.create_vector_db(texts)

Creating vector database...
Added 8 chunks for text_0
Total chunks: 8
Vector shape: (8, 219)
Vocabulary size: 219


## Perform Similarity Search

Now we query the vector database with "What is a Pokemon?" and see which chunks are most similar.

**What to expect:** Since the document is about Vector Databases (not Pokemon), the similarity scores will likely be low. But the system will still return the chunks that are *most* relevant — showing how vector search finds the best match even when the query doesn't perfectly align with the data.

The result shows:
- Which chunk IDs matched
- Their cosine similarity scores (0 to 1)
- The top 5 most similar chunks sorted by score

The similarity scores explained briefly:

- __`text_0_chunk_0 = 0.3273`__ — The document starts with *"What Is a Vector Database?"* and the query is *"What is a Pokemon?"*. TF-IDF matches the common words __"What"__, __"is"__, __"a"__ — giving ~33% overlap. This is a __word-match coincidence__, not semantic understanding.

- __`text_0_chunks 4-7 = 0.0000`__ — These chunks talk about indexing, performance, applications — __zero word overlap__ with "Pokemon", so similarity is 0.

__Key takeaway:__ TF-IDF is a __bag-of-words__ method. It only matches exact word occurrences, not meaning. "Pokemon" and "Vector Database" share no vocabulary (except stopwords), so most chunks score 0. The one non-zero match is purely from common English words in the document's title.


In [5]:
# Define a query — try changing this to something related to Vector Databases!
query_text = "What is a Pokemon?"
print(f"\nPerforming similarity search with query text: '{query_text}'")
results = search_engine.search_similar_texts(query_text)

print("\nTop 5 similar chunks:")
for chunk_id, similarity in results:
    print(f"{chunk_id}: Similarity = {similarity:.4f}")


Performing similarity search with query text: 'What is a Pokemon?'
Generated query vector for: 'What is a Pokemon?...'
[0.32730465 0.         0.         0.         0.        ]
Similarity between query and text_0_chunk_0: 0.3273
Similarity between query and text_0_chunk_7: 0.0000
Similarity between query and text_0_chunk_6: 0.0000
Similarity between query and text_0_chunk_5: 0.0000
Similarity between query and text_0_chunk_4: 0.0000

Top 5 similar chunks:
text_0_chunk_0: Similarity = 0.3273
text_0_chunk_7: Similarity = 0.0000
text_0_chunk_6: Similarity = 0.0000
text_0_chunk_5: Similarity = 0.0000
text_0_chunk_4: Similarity = 0.0000


## 🧪 Experiment: Try Different Queries

Change the `query_text` above to see how results change:

| Query | Expected Match |
|-------|---------------|
| "What is a vector database?" | First chunk (introduction) |
| "Similarity search" | Chunks about efficient similarity search / nearest neighbor |
| "E-commerce recommendations" | Chunks about product recommendations and visual search |
| "Healthcare imaging" | Chunks about medical imaging and genomics |
| "B-tree index" | Chunks comparing indexing structures |

### How This Relates to RAG

In a real RAG pipeline:
1. **Indexing**: Documents are chunked and vectorized (as shown here)
2. **Retrieval**: A query is converted to a vector and similar chunks are found
3. **Generation**: The retrieved chunks are passed as context to an LLM to generate an answer

The quality of chunking, vectorization, and retrieval directly impacts the quality of the final RAG response!

In [6]:
# Define a query — try changing this to something related to Vector Databases!
query_text = "What is Vector Database?"
print(f"\nPerforming similarity search with query text: '{query_text}'")
results = search_engine.search_similar_texts(query_text)

print("\nTop 5 similar chunks:")
for chunk_id, similarity in results:
    print(f"{chunk_id}: Similarity = {similarity:.4f}")


Performing similarity search with query text: 'What is Vector Database?'
Generated query vector for: 'What is Vector Database?...'
[0.49206886 0.02117707 0.04004578 0.04014671 0.04040685]
Similarity between query and text_0_chunk_0: 0.4921
Similarity between query and text_0_chunk_4: 0.0404
Similarity between query and text_0_chunk_3: 0.0401
Similarity between query and text_0_chunk_2: 0.0400
Similarity between query and text_0_chunk_1: 0.0212

Top 5 similar chunks:
text_0_chunk_0: Similarity = 0.4921
text_0_chunk_4: Similarity = 0.0404
text_0_chunk_3: Similarity = 0.0401
text_0_chunk_2: Similarity = 0.0400
text_0_chunk_1: Similarity = 0.0212
